# Async/Await and Asyncio

**Async programming** lets you write code that handles many I/O operations concurrently **without threads**.

Instead of blocking while waiting (e.g., for a network response), `async` code **yields control** back to the event loop, which runs other tasks.

### Key Concepts
| Concept | Meaning |
|---------|--------|
| **Event loop** | Orchestrator that runs coroutines |
| **Coroutine** | A function defined with `async def` |
| **await** | Pause here and let other tasks run |
| **Task** | A scheduled coroutine running in the event loop |

### When to Use Async
- Many concurrent HTTP requests
- Database queries
- WebSocket connections
- Reading many files

**Don't use** async for CPU-heavy work — use multiprocessing for that.

## 1. Your First Coroutine

In [ ]:
import asyncio

# async def defines a coroutine function
async def greet(name):
    print(f'Hello, {name}!')
    await asyncio.sleep(1)  # simulates I/O wait (non-blocking)
    print(f'Goodbye, {name}!')

# Run the coroutine
asyncio.run(greet('World'))

In [ ]:
import asyncio

# Coroutine that returns a value
async def fetch_data(url):
    print(f'Fetching {url}...')
    await asyncio.sleep(0.5)  # simulate network delay
    return f'Data from {url}'

async def main():
    result = await fetch_data('https://api.example.com/users')
    print(f'Got: {result}')

asyncio.run(main())

## 2. Running Multiple Coroutines Concurrently

`asyncio.gather()` runs multiple coroutines concurrently and waits for all to complete.

In [ ]:
import asyncio
import time

async def fetch(url, delay):
    print(f'  Start: {url}')
    await asyncio.sleep(delay)  # non-blocking wait
    print(f'  Done:  {url}')
    return f'Result from {url}'

async def sequential():
    start = time.time()
    r1 = await fetch('url1', 1)
    r2 = await fetch('url2', 2)
    r3 = await fetch('url3', 1)
    print(f'Sequential: {time.time()-start:.2f}s')
    return [r1, r2, r3]

async def concurrent():
    start = time.time()
    results = await asyncio.gather(
        fetch('url1', 1),
        fetch('url2', 2),
        fetch('url3', 1),
    )
    print(f'Concurrent: {time.time()-start:.2f}s')
    return results

print('=== Sequential ===')
asyncio.run(sequential())

print('\n=== Concurrent ===')
asyncio.run(concurrent())

## 3. Tasks — Background Concurrent Work

`asyncio.create_task()` schedules a coroutine to run in the background while you do other work.

In [ ]:
import asyncio

async def background_task(name, delay):
    print(f'{name}: starting')
    await asyncio.sleep(delay)
    print(f'{name}: done')
    return f'{name} result'

async def main():
    # Create tasks — they start running immediately
    task1 = asyncio.create_task(background_task('A', 2))
    task2 = asyncio.create_task(background_task('B', 1))
    task3 = asyncio.create_task(background_task('C', 3))
    
    print('Tasks created, doing other work...')
    await asyncio.sleep(0.5)
    print('Other work done, waiting for tasks...')
    
    # Wait for all tasks
    results = await asyncio.gather(task1, task2, task3)
    print('All tasks done:', results)

asyncio.run(main())

## 4. `asyncio.gather()` with Error Handling

In [ ]:
import asyncio

async def risky_task(name, should_fail=False):
    await asyncio.sleep(0.1)
    if should_fail:
        raise ValueError(f'{name} failed!')
    return f'{name} succeeded'

async def main():
    # return_exceptions=True — failures become results, not crashes
    results = await asyncio.gather(
        risky_task('task1'),
        risky_task('task2', should_fail=True),
        risky_task('task3'),
        return_exceptions=True
    )
    
    for i, result in enumerate(results, 1):
        if isinstance(result, Exception):
            print(f'Task {i} failed: {result}')
        else:
            print(f'Task {i}: {result}')

asyncio.run(main())

## 5. Async Context Managers and Iterators

In [ ]:
import asyncio

# Async context manager
class AsyncConnection:
    def __init__(self, host):
        self.host = host
    
    async def __aenter__(self):
        print(f'Connecting to {self.host}...')
        await asyncio.sleep(0.1)  # simulate connection
        print(f'Connected to {self.host}')
        return self
    
    async def __aexit__(self, *args):
        print(f'Disconnecting from {self.host}')
        await asyncio.sleep(0.05)  # simulate disconnect
    
    async def query(self, sql):
        await asyncio.sleep(0.05)
        return f'Results for: {sql}'


async def main():
    async with AsyncConnection('db.example.com') as conn:
        result = await conn.query('SELECT * FROM users')
        print(result)

asyncio.run(main())

In [ ]:
import asyncio

# Async iterator
class AsyncCounter:
    def __init__(self, stop):
        self.current = 0
        self.stop = stop
    
    def __aiter__(self):
        return self
    
    async def __anext__(self):
        if self.current >= self.stop:
            raise StopAsyncIteration
        await asyncio.sleep(0.01)
        self.current += 1
        return self.current


async def main():
    async for number in AsyncCounter(5):
        print(number, end=' ')
    print()

asyncio.run(main())

## 6. Timeouts

In [ ]:
import asyncio

async def slow_operation():
    await asyncio.sleep(5)  # takes 5 seconds
    return 'done'

async def main():
    try:
        result = await asyncio.wait_for(slow_operation(), timeout=1.0)
    except asyncio.TimeoutError:
        print('Operation timed out!')

asyncio.run(main())

## Common Mistakes

### Mistake 1: Forgetting to `await` a Coroutine

In [ ]:
import asyncio

async def get_data():
    await asyncio.sleep(0)
    return 42

async def bad_example():
    # BAD — no await: result is a coroutine object, not 42!
    result = get_data()  # forgot await!
    print(f'Bad result type: {type(result)}')  # <class 'coroutine'>
    result.close()  # need to close to avoid warning

async def good_example():
    # GOOD
    result = await get_data()
    print(f'Good result: {result}')  # 42

asyncio.run(bad_example())
asyncio.run(good_example())

### Mistake 2: Using Blocking Code Inside Async Functions

In [ ]:
import asyncio
import time

async def bad_async():
    # BAD — time.sleep() BLOCKS the entire event loop!
    time.sleep(1)  # nothing else can run during this second
    return 'done'

async def good_async():
    # GOOD — asyncio.sleep() yields control, other tasks can run
    await asyncio.sleep(1)
    return 'done'

print('Use asyncio.sleep() instead of time.sleep() in async functions')
asyncio.run(good_async())

## Mini-Project: Async Web Scraper Simulation

In [ ]:
import asyncio
import random
import time

# Simulated webpage data
PAGES = {
    'https://news.site.com/tech': ['Python 4.0 released', 'AI boom continues', 'New framework launches'],
    'https://news.site.com/sports': ['Team wins championship', 'Record broken', 'Transfer news'],
    'https://news.site.com/science': ['Mars mission update', 'Climate study published'],
    'https://news.site.com/business': ['Market hits record', 'Merger announced', 'Startup funded'],
    'https://news.site.com/health': ['New treatment found', 'Vaccine study results'],
}

async def fetch_page(url, semaphore):
    """Fetch a page with rate limiting via semaphore."""
    async with semaphore:  # limits to N concurrent requests
        delay = random.uniform(0.3, 1.5)  # simulate network variability
        print(f'  Fetching {url}...')
        await asyncio.sleep(delay)
        headlines = PAGES.get(url, ['No headlines found'])
        return {'url': url, 'headlines': headlines, 'fetch_time': delay}


async def scrape_news(urls, max_concurrent=3):
    """Scrape multiple news pages concurrently."""
    semaphore = asyncio.Semaphore(max_concurrent)  # limit concurrency
    
    tasks = [fetch_page(url, semaphore) for url in urls]
    results = await asyncio.gather(*tasks, return_exceptions=True)
    return [r for r in results if not isinstance(r, Exception)]


async def main():
    urls = list(PAGES.keys())
    print(f'Scraping {len(urls)} pages (max 3 concurrent)...\n')
    
    start = time.time()
    pages = await scrape_news(urls)
    elapsed = time.time() - start
    
    print(f'\n=== Results ({elapsed:.2f}s) ===')
    all_headlines = []
    for page in pages:
        category = page['url'].split('/')[-1]
        print(f'\n[{category.upper()}]')
        for h in page['headlines']:
            print(f'  • {h}')
            all_headlines.append(h)
    
    print(f'\nTotal headlines collected: {len(all_headlines)}')

asyncio.run(main())